# YOLO11s stability: grouped cross-validation over unseen videos

**Question.** Which fine-tuning distance from the VisDrone-DET prior, and which inference
settings, improve detection consistently across videos that the pretrained model never
saw? The Kaggle Public score plays no part in any decision here.

**Why this protocol.**

- The submitted model (`low_learning_rate` `best.pt` with test-time augmentation, clean-fold
  mAP 0.3724, Public 0.3618) was the best of about 14 configurations on one six-clip fold.
  The same fold had also chosen `best.pt`. Its gains over notebook 10 (+0.75 and +0.96
  points) are within the 1.2 to 1.6 point swing between consecutive epochs of one run, and
  the test-time augmentation gain came from one class (van, +8.4 AP).
- Public and Private are split by clip. The 48 test clips are short and differ strongly in
  class mix, so the Public part holds about 14 clips. A gain carried by a few videos or one
  class is unlikely to transfer to the other part.
- `config/visdrone_det_frame_index.json` locates every VisDrone-DET image in its raw video.
  DET train and val contain images from the raw videos of three clean-fold clips (150, 143
  and 361), 480 to 1342 frames from the clip. Only four labeled clips come from videos that
  DET train and val do not contain.
- Every fine-tuning run of notebooks 10 and 11 peaked after one epoch. A lower learning
  rate gave a higher peak and a slower decline, while a frozen backbone and twice as many
  frames per clip did not help.

**Protocol.**

- **Validation pool.** The clips of raw videos with no DET train or val frame inside any of
  their clips. A video is the independent unit; frames and clips of one video are not.
- **Folds.** Four folds keep videos whole and balance annotated objects. Each fold trains
  from the same DET initialization on every other labeled clip, every fourth frame, for two
  epochs. Native validation is off, so no checkpoint is chosen on the evaluated videos.
- **Candidates in every fold.** The initial checkpoint; the notebook 10 recipe (control)
  and the low learning rate with three seeds after one and two epochs; interpolations
  between the initial and the control epoch-1 weights; and the uniform average of the three
  low-learning-rate epoch-1 checkpoints.
- **Inference checks** on three checkpoints: multi-label NMS, test-time augmentation and
  both. A stress test shrinks frames to a 680 or 960 pixel long side before the usual
  1280 pixel resize, like the 600 low-resolution test frames.
- **Metric.** COCO mAP at 100 detections with crowd regions ignored, per video and pooled
  per fold, on every second frame of the held-out clips.

**Preregistered decision rule** (`[decision]` in `config/yolo11s_grouped_cv.toml`). A
candidate replaces the control only if all of the following hold for the paired per-video
differences:

1. the mean is positive;
2. at least 67% of videos improve;
3. the exact one-sided sign-flip p-value is at most 0.1;
4. the mean class difference stays positive without the class that gained most;
5. no video loses more than 2 mAP points.

The three primary hypotheses are fixed in advance: low learning rate seed 2026 after one
epoch, interpolation with alpha 0.5, and the low-learning-rate average. Every other
candidate is a diagnostic. We also report a reversal rate: how often a random 30% of the
videos and the remaining 70% disagree on the sign of the mean difference. It is a local
analogue of the clip-level Public/Private split and is not part of the rule.

**Running on Kaggle.**

1. Use the same setup as notebooks 10 and 11: the competition data, the `hm2-yolo-native`
   dataset rebuilt with `scripts/build_yolo_bundle.py`, GPU T4 x2 and the `HM2_DRIVE_TOKEN`
   secret.
2. Set `SHOULD_TRAIN = True` and use *Save Version → Save & Run All*.
3. Expect about 3 hours of training (16 runs of two epochs) and about 2 hours of evaluation.

Finished runs and evaluations are mirrored to Drive and skipped on a rerun. To re-render
every table from the stored records, keep `SHOULD_TRAIN = False`. Setting
`SHOULD_EXPORT_FINAL = True` also restores the final model and rewrites its submission.

## 1. Environment and configuration

The first cell is the bootstrap of notebooks 10 and 11.

In [1]:
import hashlib
import importlib
import importlib.metadata
import importlib.util
import json
import logging
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path
from zipfile import ZipFile

KAGGLE_INPUT = Path("/kaggle/input")
IS_KAGGLE = KAGGLE_INPUT.is_dir()
BUNDLE_NAME = "yolo_native_project.zip"
DRIVE_ROOT = "16wH9gNp13kfa4T039nsiFcZwYOlEJI5Z"
SHOULD_USE_DRIVE = IS_KAGGLE
PINNED_PACKAGES = {
    "ultralytics": "ultralytics==8.3.235",
    "thop": "ultralytics-thop>=2.0.18",
    "faster_coco_eval": "faster-coco-eval>=1.8.0",
}
REQUIREMENTS = {
    "polars": "polars>=1.31",
    "plotly": "plotly>=6",
    "pycocotools": "pycocotools>=2.0.11",
    "googleapiclient": "google-api-python-client",
    "google.oauth2": "google-auth",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
LOGGER = logging.getLogger("yolo_native_pipeline")

attached_bundles, wheels = [], None
if IS_KAGGLE:
    attached_bundles = sorted(
        {path for path in KAGGLE_INPUT.rglob(BUNDLE_NAME) if path.is_file()}
        | {path.parent for path in KAGGLE_INPUT.rglob("bundle_manifest.json")}
    )
    if len(attached_bundles) > 1:
        raise ValueError(f"Attach exactly one code bundle: {attached_bundles}")
    if attached_bundles:
        bundle_dataset = KAGGLE_INPUT / attached_bundles[0].relative_to(KAGGLE_INPUT).parts[0]
        wheels = next((path.parent for path in bundle_dataset.rglob("*.whl")), None)
    pinned = [
        package
        for module, package in PINNED_PACKAGES.items()
        if importlib.util.find_spec(module) is None
    ]
    if (
        "ultralytics==8.3.235" not in pinned
        and importlib.metadata.version("ultralytics") != "8.3.235"
    ):
        pinned.append("ultralytics==8.3.235")
    missing = [
        package
        for module, package in REQUIREMENTS.items()
        if importlib.util.find_spec(module) is None
    ]
    installer_prefix = []
    if wheels is not None:
        installer_prefix = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--find-links",
            str(wheels),
        ]
    elif pinned or missing:
        uv_path = shutil.which("uv")
        if uv_path is None:
            installer = Path("/kaggle/working/install_uv.sh")
            urllib.request.urlretrieve("https://astral.sh/uv/install.sh", installer)
            subprocess.run(["sh", str(installer)], check=True)
            uv_path = str(Path.home() / ".local/bin/uv")
        installer_prefix = [uv_path, "pip", "install", "--system", "--python", sys.executable]
    if pinned:
        subprocess.run([*installer_prefix, "--no-deps", *pinned], check=True)
    if missing:
        subprocess.run([*installer_prefix, *missing], check=True)
    LOGGER.info(
        "Installed packages without dependencies: %s; with dependencies: %s", pinned, missing
    )

drive_service = None
if SHOULD_USE_DRIVE:
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build

    UserSecretsClient = importlib.import_module("kaggle_secrets").UserSecretsClient
    token = json.loads(UserSecretsClient().get_secret("HM2_DRIVE_TOKEN"))
    if not token.get("refresh_token"):
        raise ValueError("HM2_DRIVE_TOKEN must contain refreshable Drive credentials.")
    drive_service = build(
        "drive",
        "v3",
        credentials=Credentials.from_authorized_user_info(token),
        cache_discovery=False,
    )
    del token

if IS_KAGGLE:
    REPOSITORY_ROOT = Path("/kaggle/temp/hm2_yolo_code")
    if attached_bundles:
        archive_path = attached_bundles[0]
    elif drive_service is not None:
        from googleapiclient.http import MediaIoBaseDownload

        project = {"id": DRIVE_ROOT}
        for name in ("inputs", BUNDLE_NAME):
            response = (
                drive_service.files()
                .list(
                    q=f"'{project['id']}' in parents and trashed = false",
                    fields="files(id,name,size,md5Checksum)",
                    pageSize=1000,
                )
                .execute(num_retries=3)
            )
            matches = [entry for entry in response["files"] if entry["name"] == name]
            if len(matches) != 1:
                raise FileNotFoundError(f"Expected exactly one {name} in the Drive inputs path.")
            project = matches[0]
        archive_path = Path("/kaggle/temp") / BUNDLE_NAME
        archive_path.parent.mkdir(parents=True, exist_ok=True)
        incoming = archive_path.with_suffix(".incoming")
        with incoming.open("wb") as stream:
            transfer = MediaIoBaseDownload(
                stream,
                drive_service.files().get_media(fileId=project["id"]),
                chunksize=4 * 1024**2,
            )
            has_completed = False
            while not has_completed:
                _, has_completed = transfer.next_chunk(num_retries=3)
        with incoming.open("rb") as stream:
            if hashlib.file_digest(stream, "md5").hexdigest() != project["md5Checksum"]:
                raise ValueError("Code bundle checksum mismatch after the Drive download.")
        incoming.replace(archive_path)
    else:
        raise FileNotFoundError(
            f"Attach a dataset containing {BUNDLE_NAME} or set SHOULD_USE_DRIVE with HM2_DRIVE_TOKEN."
        )
    shutil.rmtree(REPOSITORY_ROOT, ignore_errors=True)
    if archive_path.is_dir():
        shutil.copytree(archive_path, REPOSITORY_ROOT)
    else:
        REPOSITORY_ROOT.mkdir(parents=True)
        with ZipFile(archive_path) as archive:
            for entry in archive.infolist():
                if not (REPOSITORY_ROOT / entry.filename).resolve().is_relative_to(REPOSITORY_ROOT):
                    raise ValueError(f"Unsafe path in the code bundle: {entry.filename}")
            archive.extractall(REPOSITORY_ROOT)
    LOGGER.info("Using project code from %s", archive_path)
else:
    working_directory = Path.cwd().resolve()
    REPOSITORY_ROOT = next(
        directory
        for directory in (working_directory, *working_directory.parents)
        if (directory / "pyproject.toml").is_file() and (directory / "src/hm2").is_dir()
    )
sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

Looking in links: /kaggle/input/datasets/ke1rro/hm2-yolo-native
Processing /kaggle/input/datasets/ke1rro/hm2-yolo-native/ultralytics-8.3.235-py3-none-any.whl
Processing /kaggle/input/datasets/ke1rro/hm2-yolo-native/ultralytics_thop-2.2.5-py3-none-any.whl
Processing /kaggle/input/datasets/ke1rro/hm2-yolo-native/faster_coco_eval-1.8.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl

2026-11-14T10:48:40 [INFO] yolo_native_pipeline: Installed packages without dependencies: ['ultralytics==8.3.235', 'ultralytics-thop>=2.0.18', 'faster-coco-eval>=1.8.0']; with dependencies: []
2026-11-14T10:48:43 [INFO] yolo_native_pipeline: Using project code from /kaggle/input/datasets/ke1rro/hm2-yolo-native/yolo_native_project


In [2]:
import itertools
import tomllib

import numpy as np
import plotly.express as px
import polars as pl
import torch
import ultralytics
from IPython.display import Markdown, display
from ultralytics import YOLO
from ultralytics.utils.torch_utils import strip_optimizer

from hm2.data import CLASS_NAMES, find_dataset_root, load_tables
from hm2.detection_data import select_frames
from hm2.experiment_checkpoint import publish_snapshot, write_json
from hm2.faster_rcnn_experiments import find_competition_root
from hm2.grouped_validation import (
    assess_candidate,
    assign_video_folds,
    checkpoint_name,
    evaluate_groups,
    interpolation_name,
    planned_candidates,
    select_validation_pool,
    soup_name,
    summarize_det_overlap,
    write_coco_groups,
)
from hm2.yolo_data import prepare_yolo_dataset, verify_yolo_dataset, write_image_list_yaml
from hm2.yolo_pipeline import (
    apply_experiment,
    average_checkpoints,
    build_yolo_submission,
    coco_detections,
    epoch_checkpoints,
    evaluate_predictions,
    load_pipeline_settings,
    predict_frames,
    training_arguments,
    write_coco_subset,
)

logging.getLogger("faster_coco_eval").setLevel(logging.WARNING)
ultralytics.settings.update(
    {
        name: False
        for name in (
            "clearml",
            "comet",
            "dvc",
            "hub",
            "mlflow",
            "neptune",
            "raytune",
            "tensorboard",
            "wandb",
        )
    }
)
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_width_chars(200)

SHOULD_PREPARE_DATA = True
SHOULD_TRAIN = False
SHOULD_EVALUATE = True
SHOULD_EXPORT_FINAL = False

PLAN = tomllib.loads((REPOSITORY_ROOT / "config/yolo11s_grouped_cv.toml").read_text())
BASE = load_pipeline_settings(REPOSITORY_ROOT / "config" / PLAN["base_config"])
SETTINGS = apply_experiment(BASE, {"dataset": PLAN["dataset"]})
CV = PLAN["cross_validation"]
INFERENCE = SETTINGS["inference"]
CONTROL = PLAN["candidates"]["control"]
PRIMARY = PLAN["candidates"]["primary"]
CANDIDATE_RUNS = planned_candidates(PLAN)
RUNS = [(recipe, seed) for recipe, values in PLAN["recipes"].items() for seed in values["seeds"]]
BATCH_ID = "yolo11s_grouped_cv_seed2026"
SECONDS_PER_TRAIN_IMAGE = 0.0457
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else REPOSITORY_ROOT
TEMP_ROOT = Path("/kaggle/temp") if IS_KAGGLE else REPOSITORY_ROOT / "data"
WEIGHTS_DIRECTORY = REPOSITORY_ROOT / ("weights" if IS_KAGGLE else "data/weights")
INITIAL_WEIGHTS = WEIGHTS_DIRECTORY / BASE["model"]["model_id"]
RUN_PROJECT = WORK_ROOT / "runs" / "yolo_native" / BATCH_ID
MIRROR_ROOT = TEMP_ROOT / "drive_mirror" / BATCH_ID
EVALUATION_ROOT = RUN_PROJECT / "evaluations"
DATASET_ROOT = TEMP_ROOT / "prepared" / "yolo11s_native_1280_stride4_all_clips"
SUBMISSION_DIRECTORY = WORK_ROOT / "submissions" if IS_KAGGLE else RUN_PROJECT / "submissions"
for directory in (RUN_PROJECT, EVALUATION_ROOT, SUBMISSION_DIRECTORY, TEMP_ROOT / "prepared"):
    directory.mkdir(parents=True, exist_ok=True)
if SHOULD_TRAIN and not torch.cuda.is_available():
    raise RuntimeError("Training requires a CUDA runtime. Select a GPU accelerator.")
device = 0 if torch.cuda.is_available() else "cpu"
GPU_COUNT = torch.cuda.device_count()
for index in range(GPU_COUNT):
    try:
        torch.zeros(1, device=f"cuda:{index}").add_(1).item()
    except RuntimeError as error:
        raise RuntimeError(
            f"{torch.cuda.get_device_name(index)} cannot run this PyTorch build. Select GPU T4 x2."
        ) from error
LOGGER.info("GPUs: %s", [torch.cuda.get_device_name(index) for index in range(GPU_COUNT)])

# As in notebook 11: with fewer GPUs we keep 8 images per GPU, and Ultralytics accumulates
# gradients to its nominal batch of 64, so every optimizer step still sees 64 images.
configured_gpus = len(str(SETTINGS["training"]["device"]).split(","))
gpu_slowdown = 1.0
if 0 < GPU_COUNT < configured_gpus:
    images_per_gpu = SETTINGS["training"]["batch"] // configured_gpus
    SETTINGS["training"].update(
        device=",".join(str(index) for index in range(GPU_COUNT)),
        batch=images_per_gpu * GPU_COUNT,
    )
    gpu_slowdown = configured_gpus / GPU_COUNT
    LOGGER.warning(
        "Found %d GPU(s) instead of %d: training on device %s with batch %d.",
        GPU_COUNT,
        configured_gpus,
        SETTINGS["training"]["device"],
        SETTINGS["training"]["batch"],
    )

training_hours = (
    len(RUNS)
    * CV["folds"]
    * CV["epochs"]
    * SETTINGS["dataset"]["expected_train_frames"]
    * SECONDS_PER_TRAIN_IMAGE
    * gpu_slowdown
    / 3600
)
LOGGER.info(
    "Switches: prepare=%s, train=%s, evaluate=%s, export=%s, drive=%s; training upper bound %.1f h",
    SHOULD_PREPARE_DATA,
    SHOULD_TRAIN,
    SHOULD_EVALUATE,
    SHOULD_EXPORT_FINAL,
    drive_service is not None,
    training_hours,
)
pl.from_dicts(
    [
        {
            "Recipe": recipe,
            "Change": values["description"],
            "Seeds": str(values["seeds"]),
            "Runs": len(values["seeds"]) * CV["folds"],
            "Epochs": CV["epochs"],
        }
        for recipe, values in PLAN["recipes"].items()
    ]
)

2026-11-14T10:48:44 [INFO] numexpr.utils: NumExpr defaulting to 4 threads.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
2026-11-14T10:48:57 [INFO] yolo_native_pipeline: GPUs: ['Tesla T4', 'Tesla T4']
2026-11-14T10:48:57 [INFO] yolo_native_pipeline: Switches: prepare=True, train=False, evaluate=True, export=False, drive=True; training upper bound 2.8 h


Recipe,Change,Seeds,Runs,Epochs
str,str,str,i64,i64
"""control""","""Notebook 10 recipe, lr0 0.002""","""[2026]""",4,2
"""low_learning_rate""","""Notebook 11 low learning rate, lr0 0.0005""","""[2026, 2027, 2028]""",12,2


## 2. Validation pool and folds

We rebuild the metadata tables from the competition files and place every labeled clip in
its raw VisDrone video. A clip is *frame-clean* when no DET train or val image of its video
falls inside its frames. It is *video-clean* when DET train and val contain no image of its
video at all. A video joins the validation pool only if all its clips are frame-clean;
otherwise a contaminated sibling clip would have to train on the same flight.

In [3]:
found_root = (
    find_competition_root(KAGGLE_INPUT)
    if IS_KAGGLE
    else find_dataset_root(REPOSITORY_ROOT / "data")
)
if found_root is None:
    raise FileNotFoundError("Attach the competition data on Kaggle or extract it under data/.")
SOURCE_ROOT: Path = found_root
frames, objects, ignored = load_tables(SOURCE_ROOT)
test_frames = pl.read_csv(SOURCE_ROOT / "test_images.csv", schema_overrides={"image_id": pl.String})
det_index = json.loads((REPOSITORY_ROOT / "config" / PLAN["det_frame_index"]).read_text())
overlap = summarize_det_overlap(frames, det_index)
POOL = assign_video_folds(select_validation_pool(overlap), CV["folds"])
VIDEO_CLEAN = set(POOL.filter(pl.col("is_video_clean"))["video"])
display(
    pl.DataFrame(
        {
            "Labeled clips": [overlap.height],
            "Frame-clean clips": [int(overlap["is_frame_clean"].sum())],
            "Video-clean clips": [int(overlap["is_video_clean"].sum())],
            "Pool clips": [POOL.height],
            "Pool videos": [POOL["video"].n_unique()],
        }
    )
)
display(
    POOL.select(
        "fold",
        "video",
        "clip",
        "split",
        "frames",
        "objects",
        "det_images_same_video",
        "det_nearest_frame_gap",
        "is_video_clean",
    ).sort("fold", "video", "clip")
)
class_counts = (
    objects.select("image_key", "class_name")
    .join(frames.select("image_key", "clip"), on="image_key")
    .join(POOL.select("clip", "fold"), on="clip")
    .group_by("fold", "class_name")
    .len()
    .pivot(on="class_name", index="fold", values="len")
    .fill_null(0)
    .sort("fold")
)
class_counts.select("fold", *[name for name in CLASS_NAMES if name in class_counts.columns])

2026-11-14T10:48:57 [INFO] hm2.data: Reading train COCO metadata
2026-11-14T10:49:17 [INFO] hm2.data: Reading val COCO metadata


Labeled clips,Frame-clean clips,Video-clean clips,Pool clips,Pool videos
i64,i64,i64,i64,i64
63,21,4,20,17


fold,video,clip,split,frames,objects,det_images_same_video,det_nearest_frame_gap,is_video_clean
i64,str,str,str,i64,i64,i64,i64,bool
0,"""0000020""","""uav0000020_00406_v""","""train""",501,9359,0,null,true
0,"""0000140""","""uav0000140_01590_v""","""train""",256,13983,6,32,false
0,"""0000278""","""uav0000278_00001_v""","""train""",962,52919,0,null,true
0,"""0000323""","""uav0000323_01173_v""","""train""",426,19865,8,203,false
1,"""0000124""","""uav0000124_00944_v""","""train""",398,49578,1,747,false
1,"""0000143""","""uav0000143_02250_v""","""train""",261,16436,8,569,false
1,"""0000329""","""uav0000329_04715_v""","""train""",196,10914,14,314,false
1,"""0000363""","""uav0000363_00001_v""","""train""",462,20415,12,127,false
2,"""0000150""","""uav0000150_02310_v""","""train""",348,37892,9,480,false


fold,person,bicycle,car,van,truck,tricycle,bus,motor
i64,u32,u32,u32,u32,u32,u32,u32,u32
0,22661,1564,52098,5716,1953,6085,470,5579
1,27491,5550,50273,2780,508,2578,672,7491
2,15615,3807,41777,6083,3612,6090,1667,14574
3,28537,3271,36716,11890,2678,3444,1543,9197


## 3. Training data and fold image lists

Every labeled clip is rendered once with the notebook 10 rule: every fourth frame, a
1280-pixel long side and ignored regions painted grey. Each fold then trains on the image
list of all clips outside its held-out videos. Ultralytics requires a validation list; we
give it a few frames of the held-out clips and switch native validation off, so it runs only
once after the last epoch and never selects a checkpoint. The `all` list trains the optional
final model on every clip.

In [4]:
manifest_path = RUN_PROJECT / "dataset_manifest.json"
if SHOULD_PREPARE_DATA:
    manifest = prepare_yolo_dataset(
        SOURCE_ROOT,
        DATASET_ROOT,
        frames,
        objects,
        ignored,
        SETTINGS["dataset"],
        os.cpu_count() or 4,
    )
    write_json(manifest_path, manifest)
else:
    manifest = json.loads(manifest_path.read_text())
checks = verify_yolo_dataset(DATASET_ROOT, frames, objects, SETTINGS["dataset"], manifest)
if not checks["is_pass"].all():
    raise ValueError(f"Prepared dataset checks failed:\n{checks.filter(~pl.col('is_pass'))}")
prepared = sorted((DATASET_ROOT / "images" / "train").glob("*/*.jpg"))
DATA_YAMLS, list_rows = {}, []
for key in [*range(CV["folds"]), "all"]:
    held_out = set() if key == "all" else set(POOL.filter(pl.col("fold") == key)["clip"])
    train_images = [path for path in prepared if path.parent.name not in held_out]
    native_images = [path for path in prepared if path.parent.name in held_out] or prepared
    DATA_YAMLS[key] = write_image_list_yaml(
        RUN_PROJECT / "folds" / f"fold_{key}",
        train_images,
        native_images[: CV["native_validation_frames"]],
    )
    list_rows.append(
        {
            "Fold": str(key),
            "Training frames": len(train_images),
            "Training clips": len({path.parent.name for path in train_images}),
            "Held-out clips": len(held_out),
        }
    )
display(Markdown(f"Prepared dataset checks passed: {checks.height} of {checks.height}."))
pl.from_dicts(list_rows)

2026-11-14T10:49:21 [INFO] hm2.yolo_data: Rendering 6786 frames into /kaggle/temp/prepared/yolo11s_native_1280_stride4_all_clips with 4 threads
2026-11-14T10:51:28 [INFO] hm2.yolo_data: Prepared dataset in 128.1 s


Prepared dataset checks passed: 16 of 16.

Fold,Training frames,Training clips,Held-out clips
str,i64,i64,i64
"""0""",6248,59,4
"""1""",6455,59,4
"""2""",6255,59,4
"""3""",6228,55,8
"""all""",6786,63,0


## 4. Evaluation frames

Each fold is evaluated on every second original frame of its held-out clips, without
masking. Ignored regions enter COCO evaluation as category-specific crowd regions, as in
notebooks 10 and 11. Neighbouring frames are near duplicates (median same-object IoU 0.90
at a stride of 1 and 0.81 at 2), so the stride halves inference time without changing the
unit of inference: the video. One COCO file per video allows per-video AP.

In [5]:
evaluation_frames = pl.concat(
    [select_frames(frames, split, CV["evaluation_frame_stride"]) for split in ("train", "val")]
).join(POOL.select("clip", pl.col("video").alias("raw_video"), "fold"), on="clip")
FOLD_EVALUATION, evaluation_rows = {}, []
for fold in range(CV["folds"]):
    fold_frames = evaluation_frames.filter(pl.col("fold") == fold).sort("clip", "frame_id")
    annotation_path = RUN_PROJECT / "annotations" / f"fold{fold}.json"
    annotation_path.parent.mkdir(parents=True, exist_ok=True)
    image_ids = write_coco_subset(SOURCE_ROOT, fold_frames, annotation_path)
    groups = {
        video: [
            image_id
            for image_id, frame_video in zip(image_ids, fold_frames["raw_video"], strict=True)
            if frame_video == video
        ]
        for video in fold_frames["raw_video"].unique(maintain_order=True)
    }
    FOLD_EVALUATION[fold] = {
        "frames": fold_frames,
        "image_ids": image_ids,
        "annotation": annotation_path,
        "groups": groups,
        "group_annotations": write_coco_groups(
            annotation_path, groups, RUN_PROJECT / "annotations" / f"fold{fold}_videos"
        ),
    }
    evaluation_rows.append(
        {"Fold": fold, "Videos": len(groups), "Evaluated frames": fold_frames.height}
    )
pl.from_dicts(evaluation_rows)

Fold,Videos,Evaluated frames
i64,i64,i64
0,4,1073
1,4,659
2,4,1059
3,5,1111


## 5. Training and evaluation per fold

Folds run one after another, so an interrupted session keeps every finished fold. Each run
saves the weights after every epoch (`save_period = 1`); we strip the optimizer state and
mirror them to Drive. Derived candidates are built from those files:

- interpolations put weight `1 - alpha` on the initial checkpoint and `alpha` on the control
  epoch-1 weights;
- the average gives equal weight to the three low-learning-rate seeds after one epoch.

The initial checkpoint's person channel is pedestrian only, so interpolations partly mix the
two person definitions; per-class AP shows the effect.

Every evaluation is written to its own JSON file and skipped when it already exists.

In [6]:
drive_api, batch_folder = None, None
if drive_service is not None:
    from hm2.drive_api import DriveApi, download_current_generation, sync_checkpoint_to_drive_api

    drive_api = DriveApi(drive_service)
    batch_folder = drive_api.folder(drive_api.folder(DRIVE_ROOT, "batches"), BATCH_ID)
    download_current_generation(
        drive_api, drive_api.folder(batch_folder, "evaluations"), EVALUATION_ROOT
    )


def publish_directory(directory: Path, name: str) -> None:
    if drive_api is None or batch_folder is None:
        return
    publish_snapshot(directory)
    sync_checkpoint_to_drive_api(drive_api, directory, drive_api.folder(batch_folder, name))


def train_run(recipe: str, seed: int, fold: int | str) -> dict[int, Path]:
    name = f"{recipe}_seed{seed}_fold{fold}"
    mirror = MIRROR_ROOT / name
    if drive_api is not None and batch_folder is not None:
        download_current_generation(drive_api, drive_api.folder(batch_folder, name), mirror)
    stored = {epoch: mirror / f"epoch{epoch}.pt" for epoch in range(1, CV["epochs"] + 1)}
    if all(path.is_file() for path in stored.values()):
        return stored
    if not SHOULD_TRAIN:
        return {}
    settings = apply_experiment(
        SETTINGS, {"training": {"seed": seed, **PLAN["recipes"][recipe]["training"]}}
    )
    arguments = training_arguments(
        settings, DATA_YAMLS[fold], RUN_PROJECT / "runs", name, CV["epochs"]
    )
    started = time.monotonic()
    YOLO(str(INITIAL_WEIGHTS)).train(
        **{**arguments, "val": False, "save_period": 1, "plots": False}
    )
    LOGGER.info("%s trained in %.1f min", name, (time.monotonic() - started) / 60)
    run_directory = RUN_PROJECT / "runs" / name
    mirror.mkdir(parents=True, exist_ok=True)
    for epoch, path in epoch_checkpoints(run_directory / "weights").items():
        strip_optimizer(path)
        shutil.copy2(path, stored[epoch])
    shutil.copy2(run_directory / "results.csv", mirror / "results.csv")
    publish_directory(mirror, name)
    return stored


def build_candidates(fold: int | str, runs: dict[tuple[str, int], dict[int, Path]]) -> dict:
    available = {"initial": INITIAL_WEIGHTS}
    for (recipe, seed), checkpoints in runs.items():
        for epoch, path in checkpoints.items():
            available[checkpoint_name(recipe, seed, epoch)] = path
    derived = RUN_PROJECT / "derived" / f"fold{fold}"
    choice = PLAN["candidates"]
    if choice["interpolation_source"] in available:
        source = available[choice["interpolation_source"]]
        for alpha in choice["interpolation_alphas"]:
            path = derived / f"{interpolation_name(alpha)}.pt"
            if not path.is_file():
                average_checkpoints([INITIAL_WEIGHTS, source], [1 - alpha, alpha], path)
            available[interpolation_name(alpha)] = path
    recipe, epoch = choice["soup_recipe"], choice["soup_epoch"]
    members = [checkpoint_name(recipe, seed, epoch) for seed in PLAN["recipes"][recipe]["seeds"]]
    if all(member in available for member in members):
        path = derived / f"{soup_name(recipe, epoch)}.pt"
        if not path.is_file():
            average_checkpoints(
                [available[member] for member in members], [1 / len(members)] * len(members), path
            )
        available[soup_name(recipe, epoch)] = path
    return available


def inference_settings(variant: str) -> dict:
    prefix, _, side = variant.partition("long_side_")
    name = prefix.removesuffix("_")
    overrides = PLAN["inference_variants"][name]["inference"] if name not in ("", "default") else {}
    return {**INFERENCE, **overrides, **({"degraded_long_side": int(side)} if side else {})}


def evaluation_plan(candidates: dict) -> list[tuple[str, str]]:
    checks = PLAN["inference_checks"]
    plan = [(name, "default") for name in candidates]
    for name in [name for name in checks["checkpoints"] if name in candidates]:
        plan += [(name, variant) for variant in PLAN["inference_variants"]]
        plan += [(name, f"long_side_{side}") for side in checks["degraded_long_sides"]]
    if CONTROL in candidates:
        plan += [
            (CONTROL, f"{variant}_long_side_{side}")
            for variant in checks["stress_variants"]
            for side in checks["degraded_long_sides"]
        ]
    return plan


def evaluate_candidate(fold: int, name: str, variant: str, path: Path) -> None:
    output = EVALUATION_ROOT / f"{name}__{variant}__fold{fold}.json"
    if output.is_file():
        return
    data, settings = FOLD_EVALUATION[fold], inference_settings(variant)
    started = time.monotonic()
    predictions = predict_frames(
        YOLO(str(path)), data["frames"], SOURCE_ROOT, settings["image_sizes"][0], settings, device
    )
    seconds = time.monotonic() - started
    records = coco_detections(predictions, data["image_ids"])
    pooled = evaluate_predictions(data["annotation"], records, data["image_ids"])
    videos = evaluate_groups(data["group_annotations"], data["groups"], records)
    write_json(
        output,
        {
            "candidate": name,
            "variant": variant,
            "fold": fold,
            "checkpoint": str(path),
            "inference": settings,
            "milliseconds_per_frame": 1000 * seconds / data["frames"].height,
            "pooled": pooled["max_dets_100"],
            "pooled_map_500": pooled["max_dets_500"]["map"],
            "videos": videos.to_dicts(),
        },
    )


for fold in range(CV["folds"]):
    runs = {(recipe, seed): train_run(recipe, seed, fold) for recipe, seed in RUNS}
    candidates = build_candidates(fold, {key: value for key, value in runs.items() if value})
    if not SHOULD_EVALUATE:
        continue
    plan = evaluation_plan(candidates)
    started = time.monotonic()
    for name, variant in plan:
        evaluate_candidate(fold, name, variant, candidates[name])
    publish_directory(EVALUATION_ROOT, "evaluations")
    LOGGER.info(
        "Fold %d: %d evaluations in %.1f min", fold, len(plan), (time.monotonic() - started) / 60
    )

2026-11-14T10:52:28 [INFO] hm2.drive_api: Download control_seed2026_epoch1__default__fold0.json (0.0 MiB)
2026-11-14T10:52:29 [INFO] hm2.drive_api: Download control_seed2026_epoch1__default__fold1.json (0.0 MiB)
2026-11-14T10:52:31 [INFO] hm2.drive_api: Download control_seed2026_epoch1__default__fold2.json (0.0 MiB)
2026-11-14T10:52:32 [INFO] hm2.drive_api: Download control_seed2026_epoch1__default__fold3.json (0.0 MiB)
2026-11-14T10:52:33 [INFO] hm2.drive_api: Download control_seed2026_epoch1__long_side_680__fold0.json (0.0 MiB)
2026-11-14T10:52:34 [INFO] hm2.drive_api: Download control_seed2026_epoch1__long_side_680__fold1.json (0.0 MiB)
2026-11-14T10:52:35 [INFO] hm2.drive_api: Download control_seed2026_epoch1__long_side_680__fold2.json (0.0 MiB)
2026-11-14T10:52:37 [INFO] hm2.drive_api: Download control_seed2026_epoch1__long_side_680__fold3.json (0.0 MiB)
2026-11-14T10:52:38 [INFO] hm2.drive_api: Download control_seed2026_epoch1__long_side_960__fold0.json (0.0 MiB)
2026-11-14T10:52

## 6. Results by candidate

`CV mAP` is the mean over folds of the pooled fold mAP. The per-video columns treat every
video equally, which is closer to a clip-level leaderboard split than a pooled score
dominated by the densest videos. Rows follow the candidate plan, not the score, to avoid
reading the table as a ranking.

In [7]:
results = [json.loads(path.read_text()) for path in sorted(EVALUATION_ROOT.glob("*__fold*.json"))]
if not results:
    raise RuntimeError("No evaluations found. Train, or restore the Drive mirror, first.")
video_scores = pl.from_dicts(
    [
        {
            "candidate": result["candidate"],
            "variant": result["variant"],
            "fold": result["fold"],
            **video,
        }
        for result in results
        for video in result["videos"]
    ],
    infer_schema_length=None,
).rename({"group": "video"})
pooled_scores = pl.from_dicts(
    [
        {
            "candidate": result["candidate"],
            "variant": result["variant"],
            "fold": result["fold"],
            "map": result["pooled"]["map"],
            "ap50": result["pooled"]["ap50"],
            "ap_small": result["pooled"]["ap_small"],
            "milliseconds_per_frame": result["milliseconds_per_frame"],
            **{f"ap_{name}": result["pooled"]["per_class_ap"][name] for name in CLASS_NAMES},
        }
        for result in results
    ],
    infer_schema_length=None,
)
VIDEO_COUNT = POOL["video"].n_unique()
WORST_COUNT = max(1, VIDEO_COUNT // 4)
ORDER = {name: index for index, name in enumerate(CANDIDATE_RUNS)}


def summarize(variant: str) -> pl.DataFrame:
    per_video = (
        video_scores.filter(pl.col("variant") == variant)
        .group_by("candidate")
        .agg(
            pl.len().alias("Videos"),
            pl.col("map").mean().alias("Mean video mAP"),
            pl.col("map").std().alias("SD across videos"),
            pl.col("map").min().alias("Worst video"),
            pl.col("map").sort().head(WORST_COUNT).mean().alias("Worst-quartile mean"),
        )
    )
    pooled = (
        pooled_scores.filter(pl.col("variant") == variant)
        .group_by("candidate")
        .agg(
            pl.len().alias("Folds"),
            pl.col("map").mean().alias("CV mAP"),
            pl.col("ap50").mean().alias("AP50"),
            pl.col("ap_small").mean().alias("AP small"),
            pl.col("milliseconds_per_frame").mean().alias("ms per frame"),
        )
    )
    return (
        pooled.join(per_video, on="candidate")
        .with_columns(
            pl.col("candidate").replace_strict(ORDER, default=len(ORDER)).alias("order"),
            pl.col("candidate").is_in(PRIMARY).alias("Primary"),
        )
        .sort("order", "candidate")
        .drop("order")
    )


summarize("default")

candidate,Folds,CV mAP,AP50,AP small,ms per frame,Videos,Mean video mAP,SD across videos,Worst video,Worst-quartile mean,Primary
str,u32,f64,f64,f64,f64,u32,f64,f64,f64,f64,bool
"""initial""",4,0.285887,0.538171,0.102313,23.999965,17,0.296405,0.061585,0.161299,0.213279,false
"""control_seed2026_epoch1""",4,0.316552,0.551365,0.107331,23.148321,17,0.332184,0.068448,0.191327,0.24832,false
"""control_seed2026_epoch2""",4,0.320816,0.558149,0.098915,23.145555,17,0.329069,0.066631,0.187115,0.247846,false
"""low_learning_rate_seed2026_epoch1""",4,0.317966,0.557117,0.117795,23.117346,17,0.331651,0.068234,0.176649,0.243153,true
"""low_learning_rate_seed2026_epoch2""",4,0.324281,0.563414,0.114618,23.137593,17,0.334633,0.066325,0.191277,0.253882,false
"""low_learning_rate_seed2027_epoch1""",4,0.317974,0.557113,0.118213,23.157493,17,0.331853,0.067796,0.179118,0.244008,false
"""low_learning_rate_seed2027_epoch2""",4,0.323863,0.562435,0.114416,23.154897,17,0.334301,0.066278,0.192489,0.253115,false
"""low_learning_rate_seed2028_epoch1""",4,0.318298,0.557601,0.117838,23.115722,17,0.332003,0.067572,0.179875,0.244332,false
"""low_learning_rate_seed2028_epoch2""",4,0.323428,0.562265,0.113996,23.371146,17,0.334353,0.066539,0.191719,0.253401,false


## 7. Paired comparisons and the decision rule

Each comparison pairs the candidate and its control on the same videos. Checkpoint
candidates are compared with the control checkpoint; inference variants and the stress test
with the same checkpoint at default inference. `should_adopt` applies the preregistered rule;
the stress rows report degradation and are not candidates for adoption.

In [8]:
def compare(candidate: str, variant: str, control: str, control_variant: str) -> dict | None:
    selected = video_scores.filter(
        (pl.col("candidate") == candidate) & (pl.col("variant") == variant)
    )
    reference = video_scores.filter(
        (pl.col("candidate") == control) & (pl.col("variant") == control_variant)
    )
    paired = selected.join(reference, on=["fold", "video"], suffix="_control")
    if paired.height < VIDEO_COUNT:
        return None
    pooled = pooled_scores.filter(
        (pl.col("candidate") == candidate) & (pl.col("variant") == variant)
    ).join(
        pooled_scores.filter(
            (pl.col("candidate") == control) & (pl.col("variant") == control_variant)
        ),
        on="fold",
        suffix="_control",
    )
    class_differences = {}
    for name in CLASS_NAMES:
        values = (pooled[f"ap_{name}"] - pooled[f"ap_{name}_control"]).drop_nulls().to_numpy()
        if values.size:
            class_differences[name] = float(values.mean())
    return {
        "candidate": candidate,
        "variant": variant,
        "control": f"{control} / {control_variant}",
        **assess_candidate(
            (paired["map"] - paired["map_control"]).to_numpy(),
            class_differences,
            PLAN["decision"],
        ),
        **{f"Δ {name}": value for name, value in class_differences.items()},
    }


checkpoint_rows = [
    row
    for name in CANDIDATE_RUNS
    if name != CONTROL and (row := compare(name, "default", CONTROL, "default"))
]
variant_rows = [
    row
    for name in PLAN["inference_checks"]["checkpoints"]
    for variant in PLAN["inference_variants"]
    if (row := compare(name, variant, name, "default"))
]
ASSESSMENTS = pl.from_dicts([*checkpoint_rows, *variant_rows], infer_schema_length=None)
ASSESSMENTS.select(
    "candidate",
    "variant",
    "control",
    "mean_difference",
    "improved_share",
    "p_value",
    "worst_group_difference",
    "mean_class_difference_without_best",
    "reversal_rate",
    "should_adopt",
).with_columns(pl.col("candidate").is_in(PRIMARY).alias("primary"))

candidate,variant,control,mean_difference,improved_share,p_value,worst_group_difference,mean_class_difference_without_best,reversal_rate,should_adopt,primary
str,str,str,f64,f64,f64,f64,f64,f64,bool,bool
"""initial""","""default""","""control_seed2026_epoch1 / default""",-0.035779,0.117647,0.999802,-0.088526,-0.039384,0.0,false,false
"""control_seed2026_epoch2""","""default""","""control_seed2026_epoch1 / default""",-0.003115,0.470588,0.832054,-0.029131,0.001394,0.3487,false,false
"""low_learning_rate_seed2026_epoch1""","""default""","""control_seed2026_epoch1 / default""",-0.000532,0.470588,0.56633,-0.031012,0.000715,0.8617,false,true
"""low_learning_rate_seed2026_epoch2""","""default""","""control_seed2026_epoch1 / default""",0.002449,0.647059,0.199707,-0.033071,0.004976,0.3255,false,false
"""low_learning_rate_seed2027_epoch1""","""default""","""control_seed2026_epoch1 / default""",-0.00033,0.470588,0.544487,-0.028706,0.000735,0.9067,false,false
"""low_learning_rate_seed2027_epoch2""","""default""","""control_seed2026_epoch1 / default""",0.002117,0.647059,0.219444,-0.030797,0.005129,0.3543,false,false
"""low_learning_rate_seed2028_epoch1""","""default""","""control_seed2026_epoch1 / default""",-0.000181,0.470588,0.523804,-0.029177,0.001054,0.9503,false,false
"""low_learning_rate_seed2028_epoch2""","""default""","""control_seed2026_epoch1 / default""",0.002169,0.764706,0.22039,-0.03211,0.004809,0.3487,false,false
"""interpolation_alpha0.25""","""default""","""control_seed2026_epoch1 / default""",-0.018307,0.294118,0.995956,-0.06624,-0.02032,0.0222,false,false


In [9]:
ASSESSMENTS.select("candidate", "variant", *[f"Δ {name}" for name in CLASS_NAMES])

candidate,variant,Δ person,Δ bicycle,Δ car,Δ van,Δ truck,Δ tricycle,Δ bus,Δ motor
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""initial""","""default""",-0.072625,-0.014366,-0.043259,-0.006349,0.030365,-0.06342,-0.021114,-0.054551
"""control_seed2026_epoch2""","""default""",0.00129,0.006014,-0.000999,0.003479,0.024348,0.000837,-0.002684,0.001823
"""low_learning_rate_seed2026_epoch1""","""default""",0.006301,0.005375,0.001633,-0.010737,0.002556,-0.00107,0.005297,0.001952
"""low_learning_rate_seed2026_epoch2""","""default""",0.006787,0.007084,0.002788,-0.00398,0.027001,0.007871,0.0124,0.001882
"""low_learning_rate_seed2027_epoch1""","""default""",0.006227,0.004862,0.001575,-0.010256,0.001566,-0.000519,0.006075,0.001843
"""low_learning_rate_seed2027_epoch2""","""default""",0.006626,0.005537,0.002515,-0.00261,0.022578,0.006957,0.015156,0.001724
"""low_learning_rate_seed2028_epoch1""","""default""",0.006484,0.005324,0.001782,-0.010224,0.002596,-0.00075,0.006583,0.002169
"""low_learning_rate_seed2028_epoch2""","""default""",0.006488,0.006514,0.002369,-0.004073,0.021342,0.006951,0.013386,0.002027
"""interpolation_alpha0.25""","""default""",-0.042628,-0.000619,-0.022903,-0.002571,0.0354,-0.029376,-0.010088,-0.034054


Per-video differences against the control. Video-clean videos, absent from DET train and
val, are marked separately: if a gain appears only on videos that share a flight with DET
images, it may reflect memorized scenes rather than better transfer.

In [10]:
control_scores = video_scores.filter(
    (pl.col("candidate") == CONTROL) & (pl.col("variant") == "default")
).select("fold", "video", pl.col("map").alias("control_map"))
differences = (
    video_scores.filter(pl.col("variant") == "default")
    .join(control_scores, on=["fold", "video"])
    .with_columns(
        (pl.col("map") - pl.col("control_map")).alias("difference"),
        pl.col("video").is_in(sorted(VIDEO_CLEAN)).alias("is_video_clean"),
        pl.col("candidate").replace_strict(ORDER, default=len(ORDER)).alias("order"),
    )
    .filter(pl.col("candidate") != CONTROL)
    .sort("order", "video")
)
px.strip(
    differences,
    x="candidate",
    y="difference",
    color="is_video_clean",
    hover_data=["video", "fold", "map"],
    title=f"Per-video mAP difference from {CONTROL}",
    labels={"difference": "mAP difference", "candidate": ""},
).add_hline(y=0, line_dash="dot")

In [11]:
differences.filter(pl.col("is_video_clean") & pl.col("candidate").is_in(PRIMARY)).pivot(
    on="candidate", index="video", values="difference"
)

video,low_learning_rate_seed2026_epoch1,interpolation_alpha0.5,low_learning_rate_soup_epoch1
str,f64,f64,f64
"""0000020""",-0.010398,-0.008196,-0.010404
"""0000138""",-0.01484,0.009368,-0.013744
"""0000268""",-0.014678,-0.000881,-0.012472
"""0000278""",0.013138,-0.00113,0.012888


## 8. Diagnostics: fine-tuning distance, seeds and resolution

**Fine-tuning distance.** Alpha 0 is the initial checkpoint and alpha 1 the control after
one epoch. A curve that peaks inside the interval supports staying closer to the DET prior.

In [12]:
alphas = {"initial": 0.0, CONTROL: 1.0} | {
    interpolation_name(alpha): alpha for alpha in PLAN["candidates"]["interpolation_alphas"]
}
curve = (
    video_scores.filter((pl.col("variant") == "default") & pl.col("candidate").is_in(list(alphas)))
    .with_columns(pl.col("candidate").replace_strict(alphas).alias("alpha"))
    .sort("alpha")
)
mean_curve = curve.group_by("alpha").agg(pl.col("map").mean()).sort("alpha")
figure = px.line(
    curve,
    x="alpha",
    y="map",
    color="video",
    markers=True,
    title="mAP per video along the interpolation from the initial to the fine-tuned weights",
    labels={"alpha": "Weight of the fine-tuned checkpoint", "map": "mAP"},
)
figure.add_scatter(
    x=mean_curve["alpha"],
    y=mean_curve["map"],
    mode="lines+markers",
    name="mean",
    line={"width": 4, "color": "black"},
)

**Seed variability.** The three low-learning-rate seeds differ only in data order and
augmentation. Their spread is the noise floor for every other comparison: a candidate gain
smaller than the seed spread is not evidence. Seed-versus-seed rows apply the decision rule
to pairs that should not differ, so they also show its false-positive behaviour.

In [13]:
seed_names = [
    checkpoint_name("low_learning_rate", seed, 1)
    for seed in PLAN["recipes"]["low_learning_rate"]["seeds"]
]
seed_spread = (
    video_scores.filter((pl.col("variant") == "default") & pl.col("candidate").is_in(seed_names))
    .group_by("fold", "video")
    .agg(
        pl.len().alias("seeds"),
        pl.col("map").std().alias("SD across seeds"),
        (pl.col("map").max() - pl.col("map").min()).alias("Range across seeds"),
    )
)
display(
    seed_spread.select(
        pl.col("SD across seeds").mean().alias("Mean per-video SD across seeds"),
        pl.col("Range across seeds").max().alias("Largest per-video range"),
    )
)
pl.from_dicts(
    [
        row
        for first, second in itertools.combinations(seed_names, 2)
        if (row := compare(second, "default", first, "default"))
    ],
    infer_schema_length=None,
).select(
    "candidate",
    "control",
    "mean_difference",
    "improved_share",
    "p_value",
    "worst_group_difference",
    "should_adopt",
)

Mean per-video SD across seeds,Largest per-video range
f64,f64
0.000794,0.003226


candidate,control,mean_difference,improved_share,p_value,worst_group_difference,should_adopt
str,str,f64,f64,f64,f64,bool
"""low_learning_rate_seed2027_epoch1""","""low_learning_rate_seed2026_epoch1 / default""",0.000202,0.529412,0.287933,-0.001854,false
"""low_learning_rate_seed2028_epoch1""","""low_learning_rate_seed2026_epoch1 / default""",0.000352,0.647059,0.14373,-0.001953,false
"""low_learning_rate_seed2028_epoch1""","""low_learning_rate_seed2027_epoch1 / default""",0.00015,0.647059,0.273491,-0.001878,false


**Resolution stress.** Negative differences are the cost of a low-resolution source. Part of
it is unavoidable, because the smallest objects lose their pixels. The last rows compare
test-time augmentation with default inference at the same low resolution: Ultralytics
augmentation also evaluates downscaled copies, which may hurt more on small sources.

In [14]:
sides = PLAN["inference_checks"]["degraded_long_sides"]
stress_rows = [
    row
    for name in PLAN["inference_checks"]["checkpoints"]
    for side in sides
    if (row := compare(name, f"long_side_{side}", name, "default"))
]
stress_rows += [
    row
    for variant in PLAN["inference_checks"]["stress_variants"]
    for side in sides
    if (row := compare(CONTROL, f"{variant}_long_side_{side}", CONTROL, f"long_side_{side}"))
]
stress_rows += [
    row
    for variant in PLAN["inference_checks"]["stress_variants"]
    if (row := compare(CONTROL, variant, CONTROL, "default"))
]
pl.from_dicts(stress_rows, infer_schema_length=None).select(
    "candidate",
    "variant",
    "control",
    "mean_difference",
    "improved_share",
    "worst_group_difference",
    "reversal_rate",
)

candidate,variant,control,mean_difference,improved_share,worst_group_difference,reversal_rate
str,str,str,f64,f64,f64,f64
"""control_seed2026_epoch1""","""long_side_680""","""control_seed2026_epoch1 / default""",-0.024204,0.117647,-0.051278,0.0
"""control_seed2026_epoch1""","""long_side_960""","""control_seed2026_epoch1 / default""",-0.00553,0.294118,-0.025942,0.077
"""low_learning_rate_seed2026_epoch1""","""long_side_680""","""low_learning_rate_seed2026_epoch1 / default""",-0.017191,0.176471,-0.053784,0.0022
"""low_learning_rate_seed2026_epoch1""","""long_side_960""","""low_learning_rate_seed2026_epoch1 / default""",-0.002609,0.470588,-0.021179,0.3428
"""low_learning_rate_soup_epoch1""","""long_side_680""","""low_learning_rate_soup_epoch1 / default""",-0.017177,0.176471,-0.053458,0.0021
"""low_learning_rate_soup_epoch1""","""long_side_960""","""low_learning_rate_soup_epoch1 / default""",-0.002938,0.411765,-0.021879,0.2966
"""control_seed2026_epoch1""","""test_time_augmentation_long_side_680""","""control_seed2026_epoch1 / long_side_680""",0.008578,0.764706,-0.009901,0.0197
"""control_seed2026_epoch1""","""test_time_augmentation_long_side_960""","""control_seed2026_epoch1 / long_side_960""",0.007299,0.647059,-0.005203,0.025
"""control_seed2026_epoch1""","""test_time_augmentation""","""control_seed2026_epoch1 / default""",0.008903,0.823529,-0.004585,0.0112


## 9. Decision

Among the primary candidates that pass the rule, we keep the one whose worst video loses
least, because the goal is consistency rather than the highest average. Without a passing
primary candidate the control stays. An inference variant is adopted only if it passes the
rule on the chosen checkpoint itself.

In [15]:
def passing(rows: pl.DataFrame) -> list[dict]:
    return (
        rows.filter(pl.col("should_adopt"))
        .sort("worst_group_difference", descending=True)
        .to_dicts()
    )


primary_rows = ASSESSMENTS.filter(
    pl.col("candidate").is_in(PRIMARY) & (pl.col("variant") == "default")
)
adopted_rows = passing(primary_rows)
ADOPTED_CANDIDATE = adopted_rows[0]["candidate"] if adopted_rows else CONTROL
variant_candidates = passing(
    ASSESSMENTS.filter(
        (pl.col("candidate") == ADOPTED_CANDIDATE) & (pl.col("variant") != "default")
    )
)
ADOPTED_VARIANT = variant_candidates[0]["variant"] if variant_candidates else "default"
lines = [
    f"- `{row['candidate']}`: mean difference {row['mean_difference']:+.4f}, "
    f"{row['improved_share']:.0%} of videos improved, p = {row['p_value']:.3f}, "
    f"worst video {row['worst_group_difference']:+.4f}, adopt: {row['should_adopt']}"
    for row in primary_rows.to_dicts()
]
display(
    Markdown(
        "\n".join(
            [
                "**Primary hypotheses against the control.**",
                "",
                *lines,
                "",
                f"**Adopted checkpoint:** `{ADOPTED_CANDIDATE}`. "
                f"**Adopted inference:** `{ADOPTED_VARIANT}`.",
            ]
        )
    )
)

**Primary hypotheses against the control.**

- `low_learning_rate_seed2026_epoch1`: mean difference -0.0005, 47% of videos improved, p = 0.566, worst video -0.0310, adopt: False
- `interpolation_alpha0.5`: mean difference -0.0038, 29% of videos improved, p = 0.792, worst video -0.0415, adopt: False
- `low_learning_rate_soup_epoch1`: mean difference -0.0003, 47% of videos improved, p = 0.540, worst video -0.0305, adopt: False

**Adopted checkpoint:** `control_seed2026_epoch1`. **Adopted inference:** `multi_label`.

## 10. Optional final model and submission

With `SHOULD_EXPORT_FINAL = True` (and `SHOULD_TRAIN = True` for new runs), the runs behind
the adopted candidate are trained on every labeled clip, the candidate is rebuilt the same
way and the test set is predicted with the adopted inference. The cross-validation numbers
above describe this recipe, not this exact final model, which has no held-out videos left.

In [16]:
if SHOULD_EXPORT_FINAL:
    final_runs = {key: train_run(*key, "all") for key in CANDIDATE_RUNS[ADOPTED_CANDIDATE]}
    final_candidates = build_candidates(
        "all", {key: value for key, value in final_runs.items() if value}
    )
    if ADOPTED_CANDIDATE not in final_candidates:
        raise RuntimeError(f"Cannot build {ADOPTED_CANDIDATE}; set SHOULD_TRAIN = True.")
    final_path = final_candidates[ADOPTED_CANDIDATE]
    settings = inference_settings(ADOPTED_VARIANT)
    test_input = test_frames.rename(
        {"file_name": "relative_path", "width": "image_width", "height": "image_height"}
    )
    predictions = predict_frames(
        YOLO(str(final_path)), test_input, SOURCE_ROOT, settings["image_sizes"][0], settings, device
    )
    submission = build_yolo_submission(predictions, test_frames)
    output = SUBMISSION_DIRECTORY / f"yolo11s_grouped_cv_{ADOPTED_CANDIDATE}_{ADOPTED_VARIANT}.csv"
    incoming = output.with_suffix(".incoming")
    submission.write_csv(incoming)
    incoming.replace(output)
    with final_path.open("rb") as stream:
        checkpoint_digest = hashlib.file_digest(stream, "sha256").hexdigest()
    summary = summarize(ADOPTED_VARIANT).filter(pl.col("candidate") == ADOPTED_CANDIDATE)
    write_json(
        output.with_suffix(".metadata.json"),
        {
            "candidate": ADOPTED_CANDIDATE,
            "variant": ADOPTED_VARIANT,
            "checkpoint": str(final_path),
            "checkpoint_sha256": checkpoint_digest,
            "inference": settings,
            "cross_validation": summary.to_dicts(),
            "rows": submission.height,
        },
    )
    display(Markdown(f"Ready to upload: `{output}` ({output.stat().st_size / 1024**2:.1f} MiB)."))

## 11. Outcome

Results of the Kaggle run of 13 and 14 November 2026, from the tables above:

| Candidate | CV mAP | Mean video mAP | Worst video | Δ vs control | Videos improved |
| --- | ---: | ---: | ---: | ---: | ---: |
| Initial VisDrone-DET checkpoint | 0.2859 | 0.2964 | 0.1613 | −3.58 | 2 of 17 |
| Control: notebook 10 recipe, epoch 1 | 0.3166 | 0.3322 | 0.1913 | — | — |
| Low learning rate, epoch 1 (primary) | 0.3180 | 0.3317 | 0.1766 | −0.05 | 8 of 17 |
| Interpolation, alpha 0.5 (primary) | 0.3167 | 0.3284 | 0.1904 | −0.38 | 5 of 17 |
| Low-learning-rate average (primary) | 0.3182 | 0.3319 | 0.1789 | −0.03 | 8 of 17 |
| Control with multi-label NMS | 0.3362 | 0.3510 | 0.1997 | +1.88 | 15 of 17 |
| Control with test-time augmentation | — | — | — | +0.89 | 14 of 17 |
| Control with multi-label NMS and test-time augmentation | 0.3371 | 0.3523 | 0.1955 | +2.01 | 16 of 17 |

- **Fine-tuning helps, the recipe does not matter.** One epoch adds 3.6 points over the
  DET checkpoint. No primary candidate passes the rule: the low learning rate, the
  interpolation and the seed average are all within 0.4 points of the control. The
  0.75-point low-learning-rate gain of notebook 11 was noise from one fold.
- **Instability comes from videos, not from training randomness.** The three seeds differ by
  0.0008 mAP per video on average, while videos differ by 0.068.
- **Multi-label NMS is the one robust gain.** It passes every criterion: +1.88 points,
  p < 0.001, worst video −0.05 points, no Public/Private-style reversal. The gain comes from
  van (+7.0 AP) and truck (+5.6 AP), the classes that the single-label predictor confused
  with car. Its cost is negligible: 23 ms per frame.
- **Test-time augmentation also passes the rule, but adds little on top of multi-label NMS.**
  Combined with it, test-time augmentation adds +0.13 points (8 of 17 videos, p = 0.31) at
  1.8 times the inference time.
- **Low-resolution sources cost accuracy.** A 680-pixel source costs 2.4 points
  (low learning rate: 1.7), and test-time augmentation recovers 0.9 points of it.

The rule adopted the control recipe with multi-label NMS. Trained on every labeled clip, it
scored 0.34 on the Kaggle Public leaderboard. We chose it by cross-validation, not by Public
score.

## Limitations

- **Small pool.** The pool holds 17 videos, and only four are absent from DET train and
  val. Gains of about one point remain hard to separate from noise, and the seed spread
  above calibrates that.
- **Unknown test origin.** The pool clips come from the same VisDrone collection as
  training. The test clips may differ further: they include frames 680 and 960 pixels wide,
  which the stress test only simulates.
- **No Kaggle scorer.** We optimize a COCO proxy of the custom Kaggle metric, whose
  implementation is not published.
- **Two-epoch schedule.** Epoch 1 repeats the warmup of notebooks 10 and 11 exactly. Epoch 2
  follows a two-epoch linear decay, so it is not the same as epoch 2 of those notebooks.
- **Recipe, not model.** The final model trains on every clip, so the cross-validation
  estimates the recipe rather than that exact model.